# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides an example for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Using the Croissant schema, each data entity is referenced by its unique `@id`. Below, we list record sets, their available fields, and their `@id` values.

In [ ]:
# List available record sets
record_sets = dataset.record_sets
print("Available record sets and their @id:")
for rs in record_sets:
    print(f" - {rs['@id']} (name: {rs.get('name', '')})")

# List fields in each record set
for rs in record_sets:
    print(f"\nRecord set '{rs.get('name', '')}' (@id: {rs['@id']}):")
    fields = rs.get('fields', [])
    for field in fields:
        print(f"  - Field: {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")

### Preview the first few records in each record set
Below we print out a sample for each record set using their `@id`.

In [ ]:
# Print sample records from each record set
for rs in record_sets:
    record_set_id = rs['@id']
    print(f"\nSample records from record set '{rs.get('name', '')}' (@id: {record_set_id}):")
    records = list(dataset.records(record_set=record_set_id))
    for x in records[:3]:  # Show first 3 records
        print(json.dumps(x, indent=2))

## 3. Data Extraction
Load data from each record set into a DataFrame. All references use the unique `@id` for record sets and fields as specified in the Croissant schema.

In [ ]:
# Extract data from each record set using @id
dataframes = {}

record_set_ids = [rs['@id'] for rs in record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for record set {record_set_id}:")
    print(df.columns.tolist())
    print(f"Sample data for {record_set_id}:")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply general data processing steps: filtering records, normalizing numeric fields, and grouping data. Each operation references the appropriate field by its `@id`. Here we demonstrate these steps for one record set (the first one, for illustration), and you can adapt for other record sets as needed.

In [ ]:
# Choose the first record set for EDA
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# For demonstration, find a numeric field @id from the schema or columns
# Here, suppose the first numeric field is 'age_at_diagnosis', and its @id is 'cr:field/age_at_diagnosis'
numeric_field_id = None
fields = next(rs for rs in record_sets if rs['@id']==record_set_id)['fields']
for field in fields:
    dtype = str(field.get('dataType', ''))
    if dtype in ['schema:Integer', 'schema:Float', 'Integer', 'Float', 'Number']:
        numeric_field_id = field['@id']
        break

if numeric_field_id is None:
    print("No numeric field found in this record set.")
else:
    # Filter for numeric_field > threshold
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a non-numeric field
        group_field_id = None
        for field in fields:
            dtype = str(field.get('dataType', ''))
            if dtype in ['schema:Text', 'Text', 'Boolean', 'schema:Boolean'] and field['@id'] != numeric_field_id:
                group_field_id = field['@id']
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Field {numeric_field_id} not present in DataFrame columns.")

## 5. Visualization
Visualize the distribution of the main numeric field or relationships between key attributes. Reference fields by their `@id` as per schema.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric field if available
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Plot boxplot grouped by a group field, if available
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and processing the FAIR^2 dataset using the `mlcroissant` library. All dataset entities—record sets, fields, and columns—were referenced by their `@id`, ensuring clear linkage to the source schema.

Key steps include:
- Loading dataset metadata for context
- Exploring record set structure and contents
- Extracting tabular data for each record set
- Performing normalization and filtering on numeric fields
- Grouping records by categorical attributes
- Visualizing distributions to support further hypothesis generation

**Note**: For more advanced analysis, consult field definitions and contextual metadata via `@id` from the Croissant schema and adapt analysis to match specific medical or genetic research questions.